# Web Scraping and Vector Database Storage with ChromaDB

This notebook demonstrates how to:
1. Scrape content from websites
2. Process and chunk the text
3. Generate embeddings using SentenceTransformers
4. Store everything in a ChromaDB vector database

## Environment Variables Reference:
Create a `.env` file with these variables to customize the behavior:
```
EMBEDDING_API=all-MiniLM-L6-v2
COLLECTION_NAME=your_collection_name
HOST_NAME=localhost
PORT_NUMBER=8000
LLM_MODEL=llama3.2
SPACE_TYPE=cosine
```
## Prerequisites
Make sure you have the following packages installed:
```bash
pip install python-dotenv langchain langchain-community chromadb sentence-transformers numpy
```

In [22]:
!pip install python-dotenv langchain langchain-community chromadb sentence-transformers numpy

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached protobuf-5.29.6-cp38-abi3-macosx_10_9_universal2.whl.metadata (592 bytes)
Using cached protobuf-5.29.6-cp38-abi3-macosx_10_9_universal2.whl (418 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.35.0
    Uninstalling protobuf-7.35.0:
      Successfully uninstalled protobuf-7.35.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.


## 1. Import Required Libraries

In [23]:
%pip install -U sentence-transformers transformers tf-keras

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached protobuf-7.35.0-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
Using cached protobuf-7.35.0-cp310-abi3-macosx_10_9_universal2.whl (433 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.55.0 requires protobuf<7,>=3.20, but you have protobuf 7.35.0 which is incompatible.
googleapis-common-protos 1.70.0 requires protobuf!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.0 which is incompatible.
mlflow-skinny 3.1.4 requires protobuf<7,>=3.12.0, but you have protobuf 7.35.0 which is incompatible.
opentelemetry-proto 1.34.1 requires protobuf<6.0,>=5.0, but you have protobuf 7.35.0 which is incompatible.
Note: you may need to rest

In [24]:
%pip uninstall -y sentence-transformers transformers huggingface_hub

Found existing installation: sentence-transformers 5.5.1
Uninstalling sentence-transformers-5.5.1:
  Successfully uninstalled sentence-transformers-5.5.1
Found existing installation: transformers 5.9.0


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Uninstalling transformers-5.9.0:
  Successfully uninstalled transformers-5.9.0
Found existing installation: huggingface_hub 1.17.0
Uninstalling huggingface_hub-1.17.0:
  Successfully uninstalled huggingface_hub-1.17.0
Note: you may need to restart the kernel to use updated packages.


In [25]:
%pip install \
sentence-transformers==2.7.0 \
transformers==4.41.2 \
huggingface_hub==0.23.0 \
tf-keras

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached sentence_transformers-2.7.0-py3-none-any.whl.metadata (11 kB)
  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.23.0-py3-none-any.whl.metadata (12 kB)
  Using cached tokenizers-0.19.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.7 kB)
Using cached sentence_transformers-2.7.0-py3-none-any.whl (171 kB)
Using cached transformers-4.41.2-py3-none-any.whl (9.1 MB)
Using cached huggingface_hub-0.23.0-py3-none-any.whl (401 kB)
Using cached tokenizers-0.19.1-cp312-cp312-macosx_11_0_arm64.whl (2.4 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [sentence-transformers]sformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffuser

In [26]:
import os
from dotenv import load_dotenv
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from chromadb.config import Settings
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer
from datetime import datetime
import hashlib

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 2. Environment Configuration

Load environment variables and set default values. You can create a `.env` file with your custom settings or use the defaults.

In [29]:
# Load environment variables
load_dotenv(override=True)

# Configuration variables
embedding_api = os.environ.get('EMBEDDING_API', 'all-MiniLM-L6-v2')  # Default SentenceTransformer model
collection_name = os.environ.get('COLLECTION_NAME', "jpmorgan_collection")
host_name = os.environ.get('HOST_NAME', "localhost")  # Fixed typo from "localost"
port_number = int(os.environ.get('PORT_NUMBER', "8000"))
llm_model = os.environ.get('LLM_MODEL', "llama3.2")
space_type = os.environ.get('SPACE_TYPE', "cosine")

print(f"Configuration:")
print(f"  Embedding API: {embedding_api}")
print(f"  Collection Name: {collection_name}")
print(f"  Host: {host_name}:{port_number}")
print(f"  Distance Space: {space_type}")

Configuration:
  Embedding API: all-MiniLM-L6-v2
  Collection Name: jpmorgan_collection
  Host: localhost:8000
  Distance Space: cosine


## 3. Vector Database Functions

These functions handle the ChromaDB connection and collection management.

In [30]:
def set_vectordb_client():
    """Create and return a ChromaDB HTTP client."""
    return chromadb.HttpClient(
        host=host_name,
        port=port_number,
        settings=Settings(allow_reset=True, anonymized_telemetry=False)
    )

def deleting_collection(collection_name):
    """Delete a collection if it exists."""
    vectordb_client = set_vectordb_client()
    
    try:
        vectordb_client.delete_collection(name=collection_name)
        print(f"✅ Collection '{collection_name}' deleted successfully.")
    except Exception as e:
        print(f"ℹ️ Collection '{collection_name}' doesn't exist or already deleted.")

def get_instance_vectordb_collections():
    """Get an existing collection or create a new one."""
    vectordb_client = set_vectordb_client()
    
    try:
        collection = vectordb_client.get_collection(name=collection_name)
        print(f"✅ Retrieved existing collection: {collection_name}")
        return collection
    except Exception as e:
        print(f"🆕 Creating new collection '{collection_name}' with {space_type} distance")
        return vectordb_client.create_collection(
            name=collection_name,
            metadata={"hnsw:space": space_type}
        )

print("✅ Vector database functions defined!")

✅ Vector database functions defined!


## 4. Document Processing Functions

Functions for scraping websites, splitting text, and hashing documents.

In [31]:
def documents_scrapped(urls):
    """Scrape content from a list of URLs and split into chunks."""
    all_documents = []
    
    print(f"🌐 Starting to scrape {len(urls)} URLs...")
    
    for i, url in enumerate(urls, 1):
        try:
            print(f"  [{i}/{len(urls)}] Scraping: {url}")
            
            # Create a loader for web content
            webpage = WebBaseLoader(url).load()
            
            # Split the scraped content into chunks
            text_splitter = CharacterTextSplitter(
                chunk_size=1000,       # Desired chunk size
                chunk_overlap=200,     # With overlap
                length_function=len,   # Use length of the text as the splitting criterion
                separator=" "          # Split on spaces to enforce chunk limits
            )
            
            # Split the content and add it to the all_documents list
            chunks = text_splitter.split_documents(webpage)
            all_documents.extend(chunks)
            print(f"    ✅ Generated {len(chunks)} chunks from {url}")
            
        except Exception as e:
            print(f"    ❌ Error scraping {url}: {str(e)}")
    
    print(f"🎉 Total documents scraped: {len(all_documents)}")
    return np.array(all_documents)

def hash_document(doc, hash_algorithm='sha256'):
    """Generate a hash for a document to create unique IDs."""
    hash_function = hashlib.new(hash_algorithm)
    
    if isinstance(doc, str):
        doc = doc.encode('utf-8')
    
    hash_function.update(doc)
    return str(hash_function.hexdigest())

print("✅ Document processing functions defined!")

✅ Document processing functions defined!


## 5. Vector Storage Function

This function generates embeddings and stores documents in the vector database.

In [32]:
def persiting_documents_vector_store(documents, instance_vectordb_collection, urls):
    """Generate embeddings and persist documents to vector store."""
    
    if len(documents) == 0:
        print("⚠️ No documents to process!")
        return
    
    print(f"🤖 Generating embeddings for {len(documents)} documents...")
    
    # Generate embeddings using SentenceTransformer
    embeddings = SentenceTransformer(embedding_api).encode(
        [doc.page_content for doc in documents], 
        batch_size=32, 
        show_progress_bar=True
    )
    
    print(f"💾 Persisting documents to vector database...")
    
    document_created = 0
    
    for doc, embedding in zip(documents, embeddings):
        page_content = doc.page_content
        
        if page_content and page_content.strip():
            # Clean the content
            page_content = page_content.strip()
            page_content = ' '.join(page_content.replace('\t\n', ' ').split())
            
            # Generate unique ID
            doc_id = hash_document(page_content)
            
            # Prepare metadata
            doc_length = len(page_content)
            current_timestamp = datetime.now().isoformat()
            
            metadata = {
                "source": doc.metadata.get("source", str(urls)),
                "title": doc.metadata.get("title", "Untitled"),
                "language": doc.metadata.get("language", "en"),
                "length": doc_length,
                "timestamp": current_timestamp
            }
            
            # Add to collection
            try:
                instance_vectordb_collection.add(
                    ids=[doc_id],
                    documents=[page_content],
                    embeddings=[embedding],
                    metadatas=[metadata]
                )
                
                document_created += 1
                
                if document_created % 10 == 0:  # Progress update every 10 documents
                    print(f"  📄 Processed {document_created} documents...")
                    
            except Exception as e:
                print(f"  ❌ Error adding document {doc_id}: {str(e)}")
    
    print(f"🎉 Successfully created {document_created} documents in vector database!")
    print(f"📊 Embedding dimensions: {len(embeddings[0]) if len(embeddings) > 0 else 'N/A'}")

print("✅ Vector storage function defined!")

✅ Vector storage function defined!


## 6. Configuration and URL Setup

Define the URLs to scrape. You can modify this list as needed.

In [33]:
# Define URLs to scrape
urls = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "https://www.reuters.com/technology/",
    "https://learn.microsoft.com/en-us/azure/ai-services/"
]

print("🎯 URLs to scrape:")
for i, url in enumerate(urls, 1):
    print(f"  {i}. {url}")

🎯 URLs to scrape:
  1. https://en.wikipedia.org/wiki/Artificial_intelligence
  2. https://www.reuters.com/technology/
  3. https://learn.microsoft.com/en-us/azure/ai-services/


## 7. Execute the Pipeline

Now let's run the complete pipeline: delete existing collection, scrape documents, and store in vector database.

In [34]:
# Step 1: Delete existing collection (if any)
print("🗑️ Step 1: Deleting existing collection...")
deleting_collection(collection_name)

🗑️ Step 1: Deleting existing collection...
✅ Collection 'jpmorgan_collection' deleted successfully.


In [36]:
# Step 2: Create/get collection instance
print("\n📁 Step 2: Setting up vector database collection...")
collection = get_instance_vectordb_collections()


📁 Step 2: Setting up vector database collection...
✅ Retrieved existing collection: jpmorgan_collection


In [37]:
# Step 3: Scrape documents
print("\n🌐 Step 3: Scraping documents from URLs...")
documents = documents_scrapped(urls)


🌐 Step 3: Scraping documents from URLs...
🌐 Starting to scrape 3 URLs...
  [1/3] Scraping: https://en.wikipedia.org/wiki/Artificial_intelligence
    ✅ Generated 270 chunks from https://en.wikipedia.org/wiki/Artificial_intelligence
  [2/3] Scraping: https://www.reuters.com/technology/
    ✅ Generated 1 chunks from https://www.reuters.com/technology/
  [3/3] Scraping: https://learn.microsoft.com/en-us/azure/ai-services/
    ✅ Generated 5 chunks from https://learn.microsoft.com/en-us/azure/ai-services/
🎉 Total documents scraped: 276


In [38]:
# Step 4: Generate embeddings and store in vector database
print("\n💾 Step 4: Generating embeddings and storing in vector database...")
persiting_documents_vector_store(documents, collection, urls)


💾 Step 4: Generating embeddings and storing in vector database...
🤖 Generating embeddings for 276 documents...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

💾 Persisting documents to vector database...
  📄 Processed 10 documents...
  📄 Processed 20 documents...
  📄 Processed 30 documents...
  📄 Processed 40 documents...
  📄 Processed 50 documents...
  📄 Processed 60 documents...
  📄 Processed 70 documents...
  📄 Processed 80 documents...
  📄 Processed 90 documents...
  📄 Processed 100 documents...
  📄 Processed 110 documents...
  📄 Processed 120 documents...
  📄 Processed 130 documents...
  📄 Processed 140 documents...
  📄 Processed 150 documents...
  📄 Processed 160 documents...
  📄 Processed 170 documents...
  📄 Processed 180 documents...
  📄 Processed 190 documents...
  📄 Processed 200 documents...
  📄 Processed 210 documents...
  📄 Processed 220 documents...
  📄 Processed 230 documents...
  📄 Processed 240 documents...
  📄 Processed 250 documents...
  📄 Processed 260 documents...
  📄 Processed 270 documents...
🎉 Successfully created 276 documents in vector database!
📊 Embedding dimensions: 384


## 8. Verify the Results

Let's check what we've stored in our vector database.

In [39]:
# Get collection info
collection_info = collection.count()
print(f"📊 Collection Statistics:")
print(f"  Total documents: {collection_info}")
print(f"  Collection name: {collection.name}")

# Get a sample of documents
if collection_info > 0:
    sample_results = collection.peek(limit=3)
    print(f"\n🔍 Sample documents:")
    for i, (doc_id, document, metadata) in enumerate(zip(
        sample_results['ids'], 
        sample_results['documents'], 
        sample_results['metadatas']
    )):
        print(f"\n  Document {i+1}:")
        print(f"    ID: {doc_id[:20]}...")
        print(f"    Source: {metadata.get('source', 'Unknown')}")
        print(f"    Length: {metadata.get('length', 'Unknown')} characters")
        print(f"    Content preview: {document[:100]}...")

📊 Collection Statistics:
  Total documents: 276
  Collection name: jpmorgan_collection

🔍 Sample documents:

  Document 1:
    ID: 15d6ac95ab534afad1ec...
    Source: https://en.wikipedia.org/wiki/Artificial_intelligence
    Length: 716 characters
    Content preview: Artificial intelligence - Wikipedia Jump to content Main menu Main menu move to sidebar hide Navigat...

  Document 2:
    ID: 7d8e250ac4ec6c38ebff...
    Source: https://en.wikipedia.org/wiki/Artificial_intelligence
    Length: 753 characters
    Content preview: intelligence 1.8 General intelligence 2 Techniques Toggle Techniques subsection 2.1 Search and optim...

  Document 3:
    ID: 939897a3a9b30272c52c...
    Source: https://en.wikipedia.org/wiki/Artificial_intelligence
    Length: 768 characters
    Content preview: and harm 4.1.1 Privacy and copyright 4.1.2 Dominance by tech giants 4.1.3 Power needs and environmen...


## 9. Optional: Query the Vector Database

Test a simple similarity search to see if everything is working correctly.

In [16]:
# Example query
query_text = "financial services banking"
print(f"🔍 Searching for: '{query_text}'")

try:
    # Generate embedding for query
    model = SentenceTransformer(embedding_api)
    query_embedding = model.encode([query_text])
    
    # Search in collection
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3
    )
    
    print(f"\n📋 Top 3 similar documents:")
    for i, (doc_id, document, metadata, distance) in enumerate(zip(
        results['ids'][0], 
        results['documents'][0], 
        results['metadatas'][0],
        results['distances'][0]
    )):
        print(f"\n  Result {i+1} (Distance: {distance:.4f}):")
        print(f"    Source: {metadata.get('source', 'Unknown')}")
        print(f"    Content: {document[:200]}...")
        
except Exception as e:
    print(f"❌ Error during search: {str(e)}")

🔍 Searching for: 'financial services banking'


/opt/anaconda3/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



📋 Top 3 similar documents:

  Result 1 (Distance: 0.8719):
    Source: https://learn.microsoft.com/en-us/azure/ai-services/
    Content: Models Visual Studio Code extension Explore more Azure AI What are Foundry Tools? Microsoft Agent Framework Document Intelligence Vision Immersive Reader Training & certification AI learning and commu...

  Result 2 (Distance: 0.8919):
    Source: https://learn.microsoft.com/en-us/azure/ai-services/
    Content: Microsoft Foundry documentation | Microsoft Learn Skip to main content This browser is no longer supported. Upgrade to Microsoft Edge to take advantage of the latest features, security updates, and te...

  Result 3 (Distance: 0.9143):
    Source: https://learn.microsoft.com/en-us/azure/ai-services/
    Content: and tools Create apps with code Quickstart - Create Foundry resources Quickstart - Chat with an agent Start with an AI template Foundry Models Foundry Models sold by Azure Deployment types Get started...


## 🎉 Pipeline Complete!

You have successfully:
1. ✅ Scraped content from multiple websites
2. ✅ Split the content into manageable chunks
3. ✅ Generated embeddings using SentenceTransformers
4. ✅ Stored everything in a ChromaDB vector database
5. ✅ Verified the results with a sample query

### Next Steps:
- Experiment with different embedding models
- Try different chunking strategies
- Add more sophisticated metadata
- Implement a full RAG (Retrieval-Augmented Generation) system
